> Notebook-friendly copy of `part-I/1.5-pandas.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# 1.5) Tabular Data with pandas

pandas is the workhorse for labelled, tabular data: the `Series` (a labelled 1D array) and the `DataFrame` (named columns sharing an index). This notebook loads a small table of daily station observations — air temperature and river discharge at two stations, with realistic sensor gaps — and works through reading, selecting, time-indexing, rolling windows, grouping, and joining. The running theme is treating missing data as physical information the sensor recorded, rather than smoothing it away — exactly the choice the generated-code bug at the end gets wrong.

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/pandas_logo.svg" alt="The pandas project logo" width="500">

<em>The pandas logo, designed by Marc Garcia for the project and taken from <a href="https://commons.wikimedia.org/wiki/File:Pandas_logo.svg">Wikimedia Commons</a>; pandas itself is BSD-3-Clause licensed.</em>

**🎯 Learning objectives**

- Build and inspect Series and DataFrames, and read a CSV robustly with typed, date-parsed columns.
- Select with .loc (labels), .iloc (positions), and boolean masks; update values by assignment and by masked `.loc`.
- Use a datetime index to resample to coarser periods and compute rolling windows.
- Aggregate with groupby.
- Produce a quick plot straight from a Series or DataFrame with `.plot()`.
- Treat missing values as physical information: detect with isna, and choose ffill or interpolate deliberately.
- Combine tables with merge and concat.
- Write results to disk with pathlib paths.

## 1.5.1 Series and DataFrame

A `Series` pairs values with an index; a `DataFrame` is a collection of columns sharing one index. Each column has its own dtype.

In [ ]:
import numpy as np
import pandas as pd

# a Series: values with a labelled index
temps = pd.Series([18.2, 17.5, 19.1],
                  index=["2024-06-01", "2024-06-02", "2024-06-03"],
                  name="temp_celsius")
print(temps)

# a DataFrame: named columns sharing an index
df = pd.DataFrame({"temp_celsius": [18.2, 17.5, 19.1],
                   "discharge_m3s": [48.0, 51.2, 47.5]})
print(df)
print(df.dtypes.to_dict())

## 1.5.2 Reading a CSV Robustly

`read_csv` infers types, but for analysis you should be explicit: parse date columns to `datetime64`, and pin the dtype of key columns. First we generate an example file; csv *writing* is covered near the end.

In [ ]:
# --- generate an example data file (uses tools covered later; just setup here) ---
from pathlib import Path

In [ ]:
Path("_files").mkdir(exist_ok=True)

In [ ]:
rng = np.random.default_rng(0)
dates = pd.date_range("2024-06-01", periods=45, freq="D")
parts = []
for name, base_temp, base_q in [("BAS", 18.0, 50.0), ("LUG", 21.0, 30.0)]:
    parts.append(pd.DataFrame({
        "date": dates,
        "station": name,
        "temp_celsius": (base_temp + rng.normal(0, 1.5, 45)).round(1),
        "discharge_m3s": (base_q + rng.normal(0, 5, 45)).round(1),
    }))
raw = pd.concat(parts, ignore_index=True)
raw.loc[[3, 4, 50], "temp_celsius"] = np.nan       # simulate sensor gaps
raw.to_csv("_files/station_observations.csv", index=False)
print("wrote station_observations.csv with", len(raw), "rows")

In [ ]:
obs = pd.read_csv(
    "_files/station_observations.csv",
    parse_dates=["date"],            # -> datetime64, not object strings
    dtype={"station": "string"},     # pin the key column's type
)
print(obs.dtypes.to_dict())
print(obs.head())

## 1.5.3 Selecting: .loc, .iloc, and Boolean Masks

`.loc` selects by label, `.iloc` by integer position, and a boolean mask filters rows by condition. Combine several conditions with `&` (and) and `|` (or) — not Python's `and`/`or` from subchapter 1.2, which do not work element-wise on a Series — and wrap each condition in its own parentheses, since `&`/`|` bind more tightly than comparisons like `==` and `>`, so omitting them changes what gets evaluated.

In [ ]:
# boolean indexing: warm days at Lugano
warm_lug = obs[(obs["station"] == "LUG") & (obs["temp_celsius"] > 22.0)]
print(warm_lug[["date", "temp_celsius"]].head())

# .iloc by position, .loc by label
print(obs.iloc[0].to_dict())          # first row, by position
print(obs.loc[0, "station"])          # one cell, by label

## 1.5.4 Updating and Transforming Values

A column is created or overwritten by assignment, applied to every row at once. A boolean mask combined with `.loc` updates only the matching rows, leaving the rest untouched.

In [ ]:
# a new column, computed for every row in one vectorised expression
obs["temp_fahrenheit"] = obs["temp_celsius"] * 9 / 5 + 32

# update only the matching rows, in place
obs.loc[obs["discharge_m3s"] > 55.0, "flow_flag"] = "high_flow"

print(obs[["station", "temp_celsius", "temp_fahrenheit", "flow_flag"]].head(3))

## 1.5.5 A datetime Index: resample and Rolling Windows

Setting a `datetime` index unlocks time-aware operations. `resample` re-bins to a coarser period; `rolling` computes a moving window.

In [ ]:
# index one station by date
bas = obs[obs["station"] == "BAS"].set_index("date").sort_index()

print("monthly mean °C:", bas["temp_celsius"].resample("MS").mean().round(2).tolist())
# rolling on the gap-free discharge: the first 6 are NaN while the window fills
print("7-day rolling mean discharge (m3 s-1):")
print(bas["discharge_m3s"].rolling(window=7).mean().round(2).head(10).tolist())

## 1.5.6 groupby: Split, Apply, Combine

`groupby` splits rows by a key, applies an aggregation to each group, and combines the results.

In [ ]:
summary = obs.groupby("station").agg(
    mean_temp=("temp_celsius", "mean"),     # mean skips NaN
    max_discharge=("discharge_m3s", "max"),
    n_obs=("temp_celsius", "size"),         # size counts every row, NaN included
)
print(summary.round(2))

`nunique` counts how many distinct values a column holds; `idxmax` (or `idxmin`) returns the *label* of the row holding the maximum (minimum) value — useful directly on a groupby result like `summary` above.

In [ ]:
print("distinct stations:", obs["station"].nunique())
print("station with the warmest mean temperature:", summary["mean_temp"].idxmax())

## 1.5.7 Quick Plots with .plot()

Series and DataFrames carry a `.plot()` method that wraps matplotlib: it reads the index for the x-axis and the column names for the legend, so a first look needs no `fig, ax` boilerplate. Pass `kind=` to choose the plot type, and `ax=` to draw into axes you already created — return to the explicit figure/axes model from subchapter 1.4 whenever a plot needs more control than this gives you. For right-skewed data — many small values and a few very large ones — pass `logy=True` to put the count axis on a log scale, so a long tail does not crush everything else into one corner.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
bas["temp_celsius"].plot(ax=ax, color="tab:red")
ax.set_ylabel("temperature (°C)")
ax.set_title("Basel, daily mean temperature")
plt.show()

`kind="bar"` turns a grouped summary into a labelled bar chart in one line, reusing `summary` from the `groupby` above.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
summary["mean_temp"].plot(ax=ax, kind="bar", color="tab:blue")
ax.set_ylabel("mean temperature (°C)")
ax.set_xlabel("station")
plt.show()

Many environmental quantities are right-skewed: most values sit near a typical range, with a long tail of rare, much larger ones. A linear count axis buries that tail near zero; passing `logy=True` spreads it out so both the common and the rare values are visible in the same plot.

In [ ]:
# right-skewed data: a few very large values dominate a linear count axis
skewed = pd.Series(np.exp(rng.normal(3.0, 1.0, 500)))

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
skewed.plot(ax=axes[0], kind="hist", color="tab:gray")
axes[0].set_title("linear count axis")
skewed.plot(ax=axes[1], kind="hist", color="tab:gray", logy=True)
axes[1].set_title("logy=True")
plt.tight_layout()
plt.show()

## 1.5.8 Missing Data as Physical Information

When a sensor stops reporting, the record should show that gap, not invent a reading. pandas marks missing values as `NaN`, detects them with `isna`, and its reductions skip them by default. How you *fill* a gap is a modelling choice: `ffill` carries the last value forward (sensible for slowly varying state), `interpolate` draws a straight line between neighbours.

In [ ]:
# missing values per column — temp_celsius's gaps are genuine sensor dropouts;
# flow_flag's are a construction artifact of the .loc assignment above, which only
# wrote a string on high-flow rows and left every other row as NaN by default
print(obs.isna().sum().to_dict())

# two fills with different physical meaning
bas_temp = obs.loc[obs["station"] == "BAS", "temp_celsius"].reset_index(drop=True)
print("with gaps:  ", bas_temp.head(6).tolist())
print("ffill:      ", bas_temp.ffill().head(6).tolist())               # carry forward
print("interpolate:", bas_temp.interpolate().round(2).head(6).tolist())  # linear

**🧠 Computational-thinking fundamental: missing is not zero**

A missing value means "we do not know", which is different from any measured number — and very different from zero, a real, often common, physical reading. Encode absence as `NaN` so that reductions skip it and gaps stay visible. Filling is a deliberate decision with consequences for every statistic computed afterwards; choose the method (carry-forward, interpolation, or leaving the gap) to match the physics, and never let a tool silently substitute zero.

## 1.5.9 Combining Tables: merge and concat

`merge` joins tables on a shared key (a database-style join); `concat` stacks tables along an axis.

In [ ]:
# join station metadata onto the observations
metadata = pd.DataFrame({
    "station": pd.array(["BAS", "LUG"], dtype="string"),
    "name": ["Basel-Binningen", "Lugano"],
    "elevation_m": [316, 273],
})
merged = obs.merge(metadata, on="station", how="left")
print(merged[["date", "station", "name", "elevation_m", "temp_celsius"]].head(3))

**ℹ️ Quick exercise: a rolling discharge mean**

Index the Lugano rows by date, then compute the 7-day rolling mean of `discharge_m3s` and print its last value, rounded to one decimal.


<details>
<summary><b>✅ Solution</b></summary>

```python
lug = obs[obs["station"] == "LUG"].set_index("date").sort_index()
roll = lug["discharge_m3s"].rolling(window=7).mean()
print(round(roll.iloc[-1], 1))
```

</details>

## 1.5.10 Writing Outputs

Save results to disk with a `pathlib` path, which keeps the code OS-independent.

In [ ]:
from pathlib import Path

out_path = Path("_files/station_summary.csv")
summary.to_csv(out_path)
print("wrote", out_path.name, "-", out_path.stat().st_size, "bytes")

## *When generated code lies: filling gaps with zero*

Asked to "clean and average" a column with gaps, an assistant fills the missing values with zero and then takes the mean. The code runs, but zero is a valid temperature, so the gaps become spurious cold readings that drag the mean down.

In [ ]:
def mean_temperature(df):
    # fill missing values, then average (as an assistant returned it)
    clean = df["temp_celsius"].fillna(0)
    return clean.mean()

bas_obs = obs[obs["station"] == "BAS"]
print("buggy mean:  ", round(mean_temperature(bas_obs), 2))
print("correct mean:", round(bas_obs["temp_celsius"].mean(), 2))   # skips NaN
print("gaps filled with 0 °C:", int(bas_obs["temp_celsius"].isna().sum()))

**⚠️ Diagnosis: fillna(0) invents a measurement**

`fillna(0)` replaced unknown temperatures with 0 °C, a perfectly valid reading, so the average is biased toward zero — here by nearly a degree, with no error raised. pandas reductions already skip `NaN`, so the fill was not only wrong but unnecessary. If a gap-free series is genuinely required, `interpolate()` respects the surrounding values; zero almost never does.

In [ ]:
def mean_temperature(df):
    # reductions skip NaN already; interpolate only if a gap-free series is needed
    return df["temp_celsius"].mean()

print("fixed mean:", round(mean_temperature(bas_obs), 2))

<details>
<summary><b>🔍 Going deeper: time zones</b></summary>

A naive timestamp has no zone; localize it, then convert.

```python
idx = pd.date_range("2024-06-01", periods=3, freq="h")
aware = idx.tz_localize("UTC")          # attach UTC
local = aware.tz_convert("Europe/Zurich")   # convert to local clock time
```

Store and compute in UTC; convert to local time only for display.

</details>

<details>
<summary><b>🔍 Going deeper: multi-index</b></summary>

A hierarchical index lets one frame hold several grouping levels.

```python
multi = obs.set_index(["station", "date"]).sort_index()
print(multi.loc["BAS"].head())      # select an outer level
```

`groupby` often produces a multi-index automatically when you group by more than one key.

</details>

<details>
<summary><b>🔍 Going deeper: apply</b></summary>

`apply` runs an arbitrary function per group or per row when no built-in aggregation fits.

```python
spread = obs.groupby("station")["temp_celsius"].apply(lambda s: s.max() - s.min())
```

Prefer vectorised built-ins (`mean`, `sum`, `agg`) where they exist; `apply` is slower and should be the fallback, not the default.

</details>

<details>
<summary><b>🔍 Going deeper: Polars as a fast alternative</b></summary>

[polars](https://pola.rs/) is a newer dataframe library with a lazy, multi-threaded engine that is often much faster on large data, with a more explicit expression API.

```python
import polars as pl
df = pl.read_csv("_files/station_observations.csv")
df.group_by("station").agg(pl.col("temp_celsius").mean())
```

The concepts transfer directly from pandas; the syntax differs. It is shown for reference and not run here.

</details>

**📌 Takeaways**

- A Series is a labelled 1D array; a DataFrame is named columns sharing an index, each with its own dtype.
- Read CSVs explicitly: `parse_dates` for time columns, `dtype` for keys.
- Select with `.loc` (labels), `.iloc` (positions), and boolean masks; a datetime index enables `resample` and `rolling`.
- Assign to a column, or to a masked `.loc` selection, to derive or update values without a loop.
- `groupby` is split-apply-combine; `merge` joins on a key, `concat` stacks.
- `.plot()` wraps matplotlib for a quick line, bar, or histogram straight from a Series or DataFrame.
- Missing is not zero: detect with `isna`, rely on NaN-skipping reductions, and fill with `ffill`/`interpolate` only as a deliberate physical choice.
- Filling gaps with `fillna(0)` silently biases statistics whenever zero is a valid value.

## Summary

| Concept | Rule to remember |
|---|---|
| Series and DataFrame | A Series is a labelled 1D array; a DataFrame is named columns sharing one index. |
| Reading | Be explicit: `parse_dates` for time columns, `dtype` for key columns. |
| Selecting | `.loc` by label, `.iloc` by position, boolean masks to filter. |
| Time | A datetime index is what enables `resample` and `rolling`. |
| Deriving | Assign to a column, or to a masked `.loc` selection — no loop needed. |
| Grouping and joining | `groupby` is split-apply-combine; `merge` joins on a key, `concat` stacks. |
| Missing data | Missing is not zero: detect with `isna`, and fill only as a deliberate physical choice. |
| The zero trap | `fillna(0)` silently biases every statistic whenever zero is itself a valid value. |

## Resources

- [Python for Data Analysis, 3rd ed. — Getting Started with pandas](https://wesmckinney.com/book/pandas-basics) — McKinney; Series/DataFrame mechanics, selection, and the data-cleaning chapter that follows it.
- [pandas — Getting started](https://pandas.pydata.org/docs/getting_started/index.html) — the official task-oriented tutorials for reading, selecting, grouping, and combining data.